# Memoria Procedimental en Agentes de IA

## ?Qu? es la Memoria Procedimental?

La **memoria procedimental** en agentes de IA hace referencia a la capacidad del agente de **recordar c?mo hacer las cosas**, es decir, almacenar instrucciones o "prompts" que gu?an su comportamiento.

A diferencia de la memoria epis?dica (que recuerda eventos pasados) o la memoria sem?ntica (que recuerda hechos), la memoria procedimental define el **procedimiento** que sigue el agente para realizar tareas.

En este notebook veremos:
1. C?mo un agente carga instrucciones desde un almac?n (store)
2. C?mo optimizar autom?ticamente esas instrucciones con feedback del usuario
3. C?mo aplicar esto en un sistema multi-agente con supervisor

---
## Paso 1: Instalaci?n de dependencias

Instalamos las bibliotecas necesarias:
- **`langmem`**: Biblioteca de LangChain para gesti?n de memoria en agentes. Incluye herramientas para optimizar prompts autom?ticamente.
- **`langgraph`**: Framework para construir agentes y flujos de trabajo con grafos.

In [ ]:
%%capture cap --no-stderr
%pip install -U langmem langgraph langchain-openai python-dotenv

---
## Paso 2: Configurar la API Key de OpenAI

El archivo `.env` debe estar en la misma carpeta que este notebook (`Clase 2.2/`) y contener:
```
OPENAI_API_KEY=sk-proj-...
```
> **Nunca compartas tu API key en el chat ni en el c?digo.**

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(dotenv_path=Path().resolve() / ".env", override=True)

# Desactivar tracing y limpiar variables que redirigen a proxies
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ.pop("LANGSMITH_API_KEY", None)
os.environ.pop("OPENAI_BASE_URL", None)
os.environ.pop("OPENAI_API_BASE", None)
os.environ.pop("OPENAI_PROXY", None)

# Diagn?stico
key = os.environ.get("OPENAI_API_KEY", "")
base_url = os.environ.get("OPENAI_BASE_URL", "none")
print(f"OPENAI_API_KEY : {key[:8]}... (longitud: {len(key)})")
print(f"OPENAI_BASE_URL: {base_url}")
print(f"LANGCHAIN_TRACING_V2: {os.environ.get('LANGCHAIN_TRACING_V2')}")

---
## Paso 2: Crear el almac?n de instrucciones (InMemoryStore)

`InMemoryStore` es un almac?n de datos **en memoria** (se pierde al reiniciar el programa). Funciona como una base de datos temporal tipo clave-valor.

- `store.put(namespace, key, value)`: Guarda un valor en el almac?n.
  - **namespace**: Tupla que agrupa elementos relacionados, como una carpeta. Aqu? usamos `("instructions",)`.
  - **key**: Nombre ?nico del elemento dentro del namespace.
  - **value**: El dato a guardar (en este caso, un diccionario con el prompt inicial).

El prompt inicial es simple: `"Write good emails."` (Escribe buenos correos).

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("instructions",), key="agent_instructions", value={"prompt": "Write good emails."})

---
## Paso 3: Crear el agente con memoria procedimental

Aqu? definimos tres componentes clave:

### 3.1 Herramienta `draft_email`
Una funci?n que simula guardar un borrador de correo. El agente la usar? como herramienta disponible. En un sistema real, aqu? se conectar?a con Gmail u otro proveedor.

### 3.2 Funci?n `prompt`
Esta funci?n se ejecuta **antes de cada llamada al LLM** y:
1. Lee las instrucciones actuales desde el `store`
2. Las inyecta como mensaje de sistema al inicio de la conversaci?n

Esto es la esencia de la memoria procedimental: **el agente consulta sus instrucciones guardadas en cada interacci?n**.

### 3.3 `create_react_agent`
Crea un agente con arquitectura **ReAct** (Reasoning + Acting): el agente razona sobre qu? herramienta usar y luego act?a.

In [ ]:
from langgraph.prebuilt import create_react_agent

def draft_email(to: str, subject: str, body: str):
    """Submit an email draft."""
    return "Draft saved successfully."

def prompt(state):
    item = store.get(("instructions",), key="agent_instructions")
    instructions = item.value["prompt"]
    sys_prompt = {"role": "system", "content": f"## Instructions\n\n{instructions}"}
    return [sys_prompt] + state['messages']

agent = create_react_agent(
    "openai:gpt-4o-mini",
    prompt=prompt,
    tools=[draft_email],
    store=store
)

---
## Paso 4: Primera invocaci?n del agente

Invocamos al agente con una solicitud de correo. El agente:
1. Lee su prompt del store (`"Write good emails."`)
2. Razona sobre c?mo escribir el correo
3. Llama a la herramienta `draft_email` con los par?metros correctos

El resultado es b?sico porque el prompt inicial es muy gen?rico. No firma con ning?n nombre espec?fico ni ofrece opciones de videollamada.

In [ ]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]}
)
result['messages'][1].pretty_print()

---
## Paso 5: Optimizar el prompt autom?ticamente

### ?Qu? es `create_prompt_optimizer`?

`create_prompt_optimizer` es una funci?n de **langmem** que crea un optimizador de prompts. Este optimizador:
- Analiza la trayectoria de conversaci?n (lo que el agente hizo)
- Considera el feedback del usuario
- Reescribe el prompt para mejorar el comportamiento del agente

Es decir, **actualiza autom?ticamente las instrucciones del agente** bas?ndose en correcciones humanas. Esto es aprendizaje procedimental.

In [ ]:
from langmem import create_prompt_optimizer

optimizer = create_prompt_optimizer("openai:gpt-4o-mini")

---
## Paso 6: Aplicar el feedback al optimizador

Aqu? ocurre la magia del aprendizaje procedimental:

1. **Obtenemos el prompt actual** del store
2. **Definimos el feedback** del usuario: en este caso, queremos que:
   - El correo siempre se firme como 'William'
   - En solicitudes de reuni?n, se ofrezca Zoom o Google Meet
3. **Invocamos el optimizador** con:
   - `prompt`: El prompt actual
   - `trajectories`: Lista de tuplas `(mensajes, feedback)` ? la conversaci?n pasada + la correcci?n

El optimizador analiza qu? hizo mal el agente y reescribe el prompt para que no vuelva a ocurrir.

In [ ]:
current_prompt = store.get(("instructions",), key="agent_instructions").value["prompt"]
feedback = {"request": "Always sign off from 'William'; for meeting requests, offer to schedule on Zoom or Google Meet"}

optimizer_result = optimizer.invoke({
    "prompt": current_prompt,
    "trajectories": [(result["messages"], feedback)]
})

---
## Paso 7: Ver el prompt optimizado

El optimizador gener? un nuevo prompt mucho m?s detallado. Ahora incluye:
- Firma con 'William'
- Instrucciones espec?ficas para reuniones (Zoom/Google Meet)
- Tono profesional
- Ejemplo de correo tipo

El optimizador convirti? una correcci?n humana vaga en instrucciones claras y accionables.

In [ ]:
print(optimizer_result)

---
## Paso 8: Guardar el nuevo prompt en el store

Actualizamos el store con el prompt mejorado. La pr?xima vez que el agente sea invocado, leer? este nuevo prompt desde el store y seguir? las instrucciones actualizadas.

Esto es como **actualizar el manual de procedimientos** del agente.

In [ ]:
store.put(("instructions",), key="agent_instructions", value={"prompt": optimizer_result})

---
## Paso 9: Segunda invocaci?n ? Verificar la mejora

Invocamos al agente con **exactamente la misma solicitud** que antes. Ahora el agente deber?a:
- Ofrecer Zoom o Google Meet como opciones de reuni?n
- Firmar el correo como 'William'

El agente aprendi? de la correcci?n humana sin necesidad de reentrenamiento del modelo.

In [ ]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]}
)
result['messages'][1].pretty_print()

---
## Paso 10: Probar con un caso diferente

Ahora probamos con una solicitud de correo diferente (no es una reuni?n, sino una actualizaci?n sobre un release). El agente deber?a:
- Mantener la firma 'William' (porque est? en el prompt)
- No mencionar Zoom/Google Meet (porque no es una reuni?n)

Esto demuestra que el agente aplica las instrucciones de forma **contextual e inteligente**.

In [ ]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Let roger@langchain.dev know that the release should be later by 4:00 PM."}]}
)
result['messages'][1].pretty_print()

---
# Sistema Multi-Agente con Memoria Procedimental

Hasta ahora ten?amos un solo agente. Ahora escalaremos a un **sistema multi-agente** donde:
- Un **supervisor** decide qu? agente usar
- Un **agente de email** maneja correos
- Un **agente de redes sociales** maneja tweets

Cada agente tiene su **propio prompt** guardado en el store, y se pueden optimizar de forma independiente.

---
## Paso 11: Instalar langgraph-supervisor

`langgraph-supervisor` es una extensi?n que facilita crear un agente supervisor que orquesta a otros agentes.

In [ ]:
%%capture cap --no-stderr
%pip install -U langgraph-supervisor

---
## Paso 12: Crear el store y los dos agentes

Creamos un nuevo store con instrucciones separadas para cada agente:
- `email_agent`: Instrucciones para escribir correos
- `twitter_agent`: Instrucciones para escribir tweets

Cada agente tiene su propia funci?n `prompt_*` que lee **solo sus instrucciones** del store. Esto permite actualizarlos de forma independiente.

**Importante:** Cada agente tiene un `name` ?nico (`email_assistant`, `social_media_agent`) para que el supervisor pueda identificarlos y delegarles tareas.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

store = InMemoryStore()

store.put(("instructions",), key="email_agent", value={"prompt": "Write good emails. Repeat your draft content to the user after submitting."})
store.put(("instructions",), key="twitter_agent", value={"prompt": "Write fire tweets. Repeat the tweet content to the user upon submission."})

agent_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def draft_email(to: str, subject: str, body: str):
    """Submit an email draft."""
    return "Draft saved successfully."

def prompt_email(state):
    item = store.get(("instructions",), key="email_agent")
    instructions = item.value["prompt"]
    sys_prompt = {"role": "system", "content": f"## Instructions\n\n{instructions}"}
    return [sys_prompt] + state['messages']

email_agent = create_react_agent(
    agent_model,
    prompt=prompt_email,
    tools=[draft_email],
    store=store,
    name="email_assistant",
)

def tweet(to: str, subject: str, body: str):
    """Post a tweet."""
    return "Legendary."

def prompt_social_media(state):
    item = store.get(("instructions",), key="twitter_agent")
    instructions = item.value["prompt"]
    sys_prompt = {"role": "system", "content": f"## Instructions\n\n{instructions}"}
    return [sys_prompt] + state['messages']

social_media_agent = create_react_agent(
    agent_model,
    prompt=prompt_social_media,
    tools=[tweet],
    store=store,
    name="social_media_agent",
)

---
## Paso 13: Crear el supervisor

`create_supervisor` crea un agente que act?a como **coordinador**:
- Recibe el mensaje del usuario
- Decide qu? agente especialista debe manejarlo
- Delega la tarea al agente correcto
- Retorna la respuesta al usuario

El supervisor tambi?n usa un LLM para tomar estas decisiones de enrutamiento.

`workflow.compile(store=store)` compila el grafo completo en una aplicaci?n ejecutable, pasando el store compartido.

In [ ]:
from langgraph_supervisor import create_supervisor
from langchain_openai import ChatOpenAI

supervisor_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

workflow = create_supervisor(
    [email_agent, social_media_agent],
    model=supervisor_model,
    prompt=(
        "You are a team supervisor managing email and tweet assistants to help with correspondence."
    )
)

app = workflow.compile(store=store)

---
## Paso 14: Primera invocaci?n del sistema multi-agente

El supervisor recibe la solicitud y decide que el `email_assistant` debe manejarla.

Accedemos a `result["messages"][3]` porque los mensajes incluyen:
- `[0]`: Mensaje del usuario
- `[1]`: Decisi?n del supervisor (delegar al email_assistant)
- `[2]`: Llamada a herramienta (draft_email)
- `[3]`: Respuesta final del email_assistant al usuario

In [ ]:
result = app.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]},
)

result["messages"][3].pretty_print()

---
## Paso 15: Optimizador multi-prompt

`create_multi_prompt_optimizer` es la versi?n avanzada del optimizador que puede:
- Manejar **m?ltiples prompts simult?neamente** (uno por agente)
- Aplicar el feedback **solo al agente relevante**
- Mantener sin cambios los prompts de agentes que no necesitan mejora

Esto es crucial en sistemas multi-agente: no queremos que el feedback sobre correos afecte las instrucciones del agente de tweets.

In [ ]:
from langmem import create_multi_prompt_optimizer

feedback = {"request": "Always sign off emails from 'William'; for meeting requests, offer to schedule on Zoom or Google Meet"}

optimizer = create_multi_prompt_optimizer("openai:gpt-4o-mini")

---
## Paso 16: La clase `Prompt` de langmem

`Prompt` es un `TypedDict` (diccionario con tipos definidos) que estructura la informaci?n de cada prompt para el optimizador:

| Campo | Obligatorio | Descripci?n |
|-------|-------------|-------------|
| `name` | S? | Identificador ?nico del prompt |
| `prompt` | S? | El texto actual del prompt |
| `update_instructions` | No | Gu?as para modificar el prompt |
| `when_to_update` | No | Condici?n para actualizar este prompt |

El campo `when_to_update` es especialmente ?til en sistemas multi-agente: le dice al optimizador **cu?ndo** debe o no debe modificar un prompt espec?fico.

In [ ]:
from langmem import Prompt
?Prompt

---
## Paso 17: Estructurar y optimizar los prompts de los agentes

Preparamos los prompts de ambos agentes como diccionarios con la estructura `Prompt`:

- **`tweet_prompt`**: Solo se actualiza si la generaci?n de tweets necesita mejora.
- **`email_prompt`**: Solo se actualiza si el feedback indica que el email necesita mejorar.

El optimizador analizar? la trayectoria de la conversaci?n y el feedback para decidir:
- ?Cu?l prompt necesita cambios?
- ?Qu? cambios espec?ficos aplicar?

En este caso, el feedback es sobre correos ? solo deber?a cambiar `email_prompt`.

In [ ]:
# Obtener los prompts actuales del store
email_prompt = store.get(("instructions",), key="email_agent").value['prompt']
tweet_prompt = store.get(("instructions",), key="twitter_agent").value['prompt']

# Estructurar como objetos Prompt con condiciones de actualizaci?n
email_prompt = {
    "name": "email_prompt",
    "prompt": email_prompt,
    "when_to_update": "Only if feedback is provided indicating email writing performance needs improved."
}
tweet_prompt = {
    "name": "tweet_prompt",
    "prompt": tweet_prompt,
    "when_to_update": "Only if tweet writing generation needs improvement."
}

# Invocar el optimizador multi-prompt
optimizer_result = optimizer.invoke({
    "prompts": [tweet_prompt, email_prompt],
    "trajectories": [(result["messages"], feedback)]
})

---
## Paso 18: Ver el resultado de la optimizaci?n multi-prompt

El resultado muestra los dos prompts:
- **`tweet_prompt`**: Sin cambios (el feedback no mencion? tweets)
- **`email_prompt`**: Actualizado con instrucciones detalladas sobre firma y opciones de videollamada

Esto demuestra la inteligencia del optimizador: aplica los cambios **solo donde son necesarios**.

In [ ]:
optimizer_result

---
## Paso 19: Actualizar el store del agente de email

Guardamos el prompt mejorado del agente de email en el store. Usamos `optimizer_result[1]` porque el resultado es una lista donde:
- `[0]` = tweet_prompt (sin cambios)
- `[1]` = email_prompt (actualizado)

El agente de redes sociales no necesita actualizaci?n.

In [ ]:
store.put(("instructions",), key="email_agent", value={"prompt": optimizer_result[1]['prompt']})

---
## Paso 20: Invocaci?n final ? Verificar la mejora en el sistema multi-agente

Probamos el sistema completo con la misma solicitud. Ahora el email_assistant deber?a:
- Ofrecer Zoom o Google Meet como opciones
- Firmar como 'William'

El supervisor sigue funcionando igual, pero el agente que delega ahora tiene mejores instrucciones.

In [ ]:
result = app.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]},
)

result["messages"][3].pretty_print()

In [ ]:
print(result['messages'][3].content)

---
# Resumen: ?Qu? aprendimos?

## Conceptos clave

| Concepto | Descripci?n |
|----------|-------------|
| **Memoria Procedimental** | El agente almacena y consulta instrucciones sobre *c?mo* realizar tareas |
| **InMemoryStore** | Almac?n temporal clave-valor para guardar prompts e instrucciones |
| **Prompt Injection din?mica** | La funci?n `prompt()` inyecta las instrucciones del store antes de cada llamada al LLM |
| **Optimizaci?n de Prompts** | `create_prompt_optimizer` actualiza autom?ticamente las instrucciones bas?ndose en feedback humano |
| **Multi-Prompt Optimizer** | `create_multi_prompt_optimizer` gestiona m?ltiples prompts y actualiza solo los relevantes |
| **Supervisor Pattern** | Un agente coordinador delega tareas a agentes especialistas |

## Flujo completo

```
Usuario ? Supervisor ? Agente especialista ? Lee prompt del store ? LLM ? Respuesta
                                                                         ?
                                               Feedback ? Optimizador ? Store actualizado
```

## ?Por qu? es importante?

La memoria procedimental permite que los agentes **mejoren con el tiempo** sin reentrenar el modelo base. Es una forma eficiente y escalable de personalizar el comportamiento de agentes de IA en aplicaciones reales.